# Pulse generator quick test

Run the cells in order with the matching FPGA bitstream, API library and SCPI server. The pulse appears on E1 pin 5 (DIO1_P); use E1 pin 25 or 26 as ground. This test uses a 1 MHz, 50% duty cycle pulse and leaves the output OFF (claimed LOW). Check the physical waveform with an oscilloscope.

If a pulse command reports **9516, Unsupported Feature**, the installed API did not find the expected pulse register ID 0x50554c01 and 2 ns tick at 0x40720000. Check that the matching full-feature FPGA image is loaded (correct top, Z20_G2, no SCOPE_ONLY) and that the updated API/server were installed. The notebook cannot fix that deployment mismatch.


In [26]:
import redpitaya_scpi as scpi

RP_IP = "169.254.77.151"  # Change to your Red Pitaya address if needed.
rp = scpi.scpi(RP_IP, port=5000, timeout=20.0)


In [27]:
def check_scpi_error(message):
    error = rp.txrx_txt("SYST:ERR:NEXT?").strip()
    if error.startswith("0,"):
        return
    if "Unsupported Feature" in error:
        raise RuntimeError(
            f"{message}: {error}. Pulse FPGA registers were not recognized; "
            "check the loaded bitstream and matching API/server."
        )
    raise RuntimeError(f"{message}: {error}")

def command(message):
    rp.tx_txt(message)
    check_scpi_error(message)

def query(message):
    response = rp.txrx_txt(message).strip()
    check_scpi_error(message)
    return response


In [ ]:
# Run this cell to generate a pulse and inspect SCPI readbacks.
# OUTPUT OFF is a write command, so it reports unsupported hardware promptly.
# If this fails, stop here; there is no pulse hardware for cleanup to control.
command('PULSE:OUTPUT OFF')

try:
    pin = query('PULSE:PIN?')
    assert pin == 'DIO1_P', f'Unexpected pulse pin: {pin}'
    command('PULSE:TIMING 1000,500')  # Period 1000 ns, HIGH 500 ns.
    assert query('PULSE:TIMING?') == '1000,500'
    assert abs(float(query('PULSE:FREQUENCY?')) - 1_000_000) < 1

    command('PULSE:OUTPUT ON')
    assert query('PULSE:OUTPUT?') == '1'
    print("Pulse running: 1 MHz, 500 ns HIGH, E1 pin 5")
    print(f"Status: {query('PULSE:STATUS?')}")
    input("Inspect the waveform, then press Enter to turn the pulse OFF...")
except BaseException:
    # Preserve the first failure if the cleanup command also fails.
    try:
        command("PULSE:OUTPUT OFF")
    except Exception as stop_error:
        print(f"Could not confirm pulse OFF: {stop_error}")
    raise
else:
    command("PULSE:OUTPUT OFF")
    assert query("PULSE:OUTPUT?") == "0"
    print("Pulse OFF; E1 pin 5 held LOW")


RuntimeError: PUSE:OUTPUT OFF: -113,"Undefined header;PUSE:OUTPUT OFF"